# Preprocess RNAseq

This notebook takes UEA featureCounts files and generates LINCS Level 5 equivalent data output.

# Imports

In [19]:
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
import os
import pandas as pd
from pathlib import Path
import pickle
from scipy.stats import spearmanr
from tqdm import tqdm as tqdm

# Directories

In [3]:
# Define current directory
cwd = Path.cwd()
# Define DATA directory
DATA = cwd.parents[1]/'data'/'canada'

# Define INPUT directory
INPUT = DATA / 'input'
# Define COUNTS directory
COUNTS = INPUT / 'featureCounts'

# Define OUTPUT directory
OUTPUT = DATA / 'output'
# Define MERGED directory
MERGED = OUTPUT / 'merged_counts'
# Define RPKM directory
RPKM = OUTPUT / 'rpkm'
# Define LEVEL3 directory
LEVEL3 = OUTPUT / 'level_3'
# Define LEVEL4 directory
LEVEL4 = OUTPUT / 'level_4'
# Define LEVEL5 directory
LEVEL5 = OUTPUT / 'level_5'

# Functions

## General

In [ ]:
def file_to_list(path):
    '''
    Converts a .txt file to a list
    '''

    with open(f'{path}', 'r', encoding = 'utf-8') as f:
        list_file = [line.strip() for line in f]
    
    return list_file

def list_to_file(path, data):
      '''
      Saves a list or set to a .txt file with no header.
      '''

      with open(path, 'w') as f:
            for item in sorted(data):
                  f.write(f'{item}\n')

def pickle_load(path: str, report: bool = False):
    '''
    Loads pickled data.
    '''

    with open(path, 'rb') as f:
        data = pickle.load(f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_load')
            print(f'Pickled graph loaded w/ {num_nodes:,} nodes and {num_edges:,} edges')
            print()
        else:
            print('>> pickle_load')
            print(f'Pickled file loaded')
            print()

    return data

def pickle_save(path: str, data, report: bool = False):
    '''
    Pickles data.
    '''

    with open(path, 'wb') as f:
        pickle.dump(data, f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_save')
            print(f'Graph w/ {num_nodes:,} nodes and {num_edges:,} edges pickled')
        else:
            print('>> pickle_save')
            print(f'Data pickled')
            print()

## Normalisation

In [5]:
def quantile_normalise(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Performs quantile normalisation on dataframe with 'geneid' index column and numeric values.
    '''

    # Convert data to array
    data = df.to_numpy(dtype = float)

    # Sort values within each column
    sorted_data = np.sort(data, axis = 0)
    # Get index order for sorted columns
    sorted_index = np.argsort(data, axis = 0)

    # Compute mean across each row (all values in same sorted rank)
    mean_values = np.mean(sorted_data, axis = 1)

    # Generate array of zero values
    data_normalised = np.zeros_like(data)

    # Iterate through columns
    for column in range(data.shape[1]):
        # Extract index values for column
        column_index = sorted_index[:, column]
        # Map mean values to genes
        data_normalised[column_index, column] = mean_values

    # Generate dataframe
    df_normalised = pd.DataFrame(data_normalised, index = df.index, columns = df.columns)

    return df_normalised

# RNAseq

## featureCounts

In [53]:
# Define list of featureCounts files
list_featurecounts = [file for file in os.listdir(COUNTS) if 'summary' not in file]
# Define set of unique samples
set_samples = set([file.split('_L00')[0] for file in list_featurecounts])
# Report
print(f'{len(set_samples):,} unique samples found')

96 unique samples found


### Extract Gene Lengths

Extract `geneid` and `length` data from each featureCounts file.

In [ ]:
# Initialise dataframe
df_featurecounts = pd.DataFrame()

# Iterate through featureCounts files
for file in tqdm(list_featurecounts, desc = 'Extracting gene lengths', total = len(list_featurecounts)):
    # Load data
    df = pd.read_csv(COUNTS / file, sep = '\t', comment = '#')
    # Extract data
    df = df[['Geneid', 'Length']]
    # Rename column
    df.rename(columns = {'Length' : file}, inplace = True)
    # Check length of df_featurecounts
    if len(df_featurecounts) > 1:
        df_featurecounts = pd.merge(df_featurecounts, df, how = 'left', on = 'Geneid')
    else:
        df_featurecounts = pd.concat([df_featurecounts, df])

# Rename column
df_featurecounts.rename(columns = {'Geneid' : 'geneid'}, inplace = True)
# Melt data
df_featurecounts = pd.melt(df_featurecounts, id_vars = ['geneid'], var_name = 'filename', value_name = 'length')
# Show data
df_featurecounts.head()

Extracting gene lengths: 100%|██████████| 192/192 [01:48<00:00,  1.77it/s]


,geneid,filename,length
0,DDX11L1,R1430-S0001_ConParo6h1_A104295_1_22WF7FLT3_TTA...,1652
1,WASH7P,R1430-S0001_ConParo6h1_A104295_1_22WF7FLT3_TTA...,1769
2,MIR6859-1,R1430-S0001_ConParo6h1_A104295_1_22WF7FLT3_TTA...,68
3,MIR1302-2HG,R1430-S0001_ConParo6h1_A104295_1_22WF7FLT3_TTA...,2263
4,MIR1302-2,R1430-S0001_ConParo6h1_A104295_1_22WF7FLT3_TTA...,138


Check for any genes w/ multiple length values in one or more files.

In [41]:
# Copy data
df_group = df_featurecounts.copy()
# Group data to check for number of unique length values/gene
df_group = df_group[['geneid', 'length']].groupby(by = 'geneid').count()
# Check for values > 192
df_slice = df_group[df_group['length'] > 192]
num_genes = len(df_slice)
print(f'{num_genes:,} genes found with multiple length values')

# Show data
df_group[df_group['length'] > 192].head()

0 genes found with multiple length values


,length
geneid,


Save gene length values

In [44]:
# Extract data
df_lengths = df_featurecounts[['geneid', 'length']].copy()
# Drop duplicates
df_lengths.drop_duplicates(inplace = True)
# Save data
df_lengths.to_csv(OUTPUT / 'df_gene_lengths.csv', index = False)
# Show data
df_lengths.head()

,geneid,length
0,DDX11L1,1652
1,WASH7P,1769
2,MIR6859-1,68
3,MIR1302-2HG,2263
4,MIR1302-2,138


### Merge Lane Files

In [58]:
# Iterate through unique samples
for sample in tqdm(set_samples, desc = 'Merging lane files', total = len(set_samples)):

    # Get sample info
    info = sample.split('_')[1]
    # Get replicate number
    replicate = info[-1]
    # Get sample description
    desc = info[:-1]

    # Load lane files
    df1 = pd.read_csv(COUNTS / f'{sample}_L001_featureCounts.txt', sep = '\t', comment = '#')
    df2 = pd.read_csv(COUNTS / f'{sample}_L002_featureCounts.txt', sep = '\t', comment = '#')

    # Extract geneid and count columns
    df1 = df1.iloc[:, [0, -1]]
    df2 = df2.iloc[:, [0, -1]]

    # Renane columns
    df1.rename(columns = {df1.columns[1] : 'lane1'}, inplace = True)
    df2.rename(columns = {df2.columns[1] : 'lane2'}, inplace = True)
    
    # Merge data
    df = pd.merge(df1, df2, how = 'left', on = 'Geneid')
    
    # Generate sum column
    df['count'] = df['lane1'] + df['lane2']
    # Extract data
    df = df[['Geneid', 'count']]
    # Rename columns
    df.rename(columns = {'Geneid' : 'geneid'}, inplace = True)

    # Save data
    df.to_csv(MERGED / f'{desc}_{replicate}.csv', index = False)

# Show example data
df.head()

Merging lane files: 100%|██████████| 96/96 [01:46<00:00,  1.11s/it]


,geneid,count
0,DDX11L1,0
1,WASH7P,271
2,MIR6859-1,7
3,MIR1302-2HG,0
4,MIR1302-2,0


## RPKM

### Calculate RPKM Values

Timepoint-specific RPKM calculated from featureCounts data using the formula:
$$
RPKM = \frac{Gene Count \space \times \space 10^6}{Length \space \times \space Total Mapped Reads}
$$

In [3]:
# Define timepoint list
list_timepoints = ['6h', '24h']

In [ ]:
# Load length data
df_lengths = pd.read_csv(OUTPUT / 'df_gene_lengths.csv')

# Iterate through timepoints
for timepoint in tqdm(list_timepoints, desc = 'Calculating RPKM values', total = len(list_timepoints)):

    # Initialise timepoint dataframe
    df_timepoint = pd.DataFrame()
    # Initialise rpkm dataframe
    df_rpkm = pd.DataFrame()

    # Get control files (uninfected control)
    list_control = [file for file in os.listdir(MERGED) if 'UI' in file]
    # Get treatment files
    list_treatment = [
        file for file in os.listdir(MERGED) if
        # Timepoint data
        timepoint in file and
        # Ignore contaminated files
        'Con' not in file and
        # Ignore 96h files (contains '6h')
        '96h_' not in file and
        # Ignore vehicle controls
        'Ve' not in file and
        # Ignore infected positive controls
        'Positive' not in file and
        # Ignore drugged positive controls
        'Colo' not in file
    ]

    # Combine file lists
    list_files = list_control + list_treatment

    # Iterate through list_files
    for file in list_files:

        # Get description
        desc = file.split('.')[0]
        # Load data
        df = pd.read_csv(MERGED / file)
        # Rename column
        df.rename(columns = {'count' : desc}, inplace = True)

        # Merge data
        if len(df_timepoint) > 0:
            df_timepoint = pd.merge(df_timepoint, df, how = 'left', on = 'geneid')
        else:
            df_timepoint = pd.concat([df_timepoint, df])
    
    # Merge with length data
    df_timepoint = pd.merge(df_timepoint, df_lengths, how = 'left', on = 'geneid')

    # Generate kilobase length column
    df_timepoint['length_kb'] = df_timepoint['length'] / 1000
    # Get gene length (kb) values
    list_length_kb = df_timepoint['length_kb'].values

    # Define count columns
    list_count_columns = df_timepoint.columns[1:-2]
    # Calculate total mapped reads per sample
    library_sizes = df_timepoint[list_count_columns].sum(axis = 0)
    # Compute RPKM
    df_rpkm = df_timepoint[list_count_columns].div(list_length_kb, axis = 0).div(library_sizes / 1e6, axis = 1)
    # Insert geneids
    df_rpkm.insert(0, 'geneid', df_timepoint['geneid'])
    # Save data
    df_rpkm.to_csv(RPKM / f'{timepoint}_rpkm.csv', index = False)

# Show example data
df_rpkm.head()

Calculating RPKM values: 100%|██████████| 2/2 [00:01<00:00,  1.32it/s]


,geneid,UI_1,UI_2,UI_3,Halo24h_1,Halo24h_2,Halo24h_3,Nita24h_1,Nita24h_2,Nita24h_3,Paro24h_1,Paro24h_2,Paro24h_3
0,DDX11L1,0.000000,0.000000,0.000000,0.000000,0.108660,0.024867,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,WASH7P,7.306364,5.610392,6.404163,20.126001,17.791656,18.647755,6.749169,6.354305,7.003300,6.490892,7.320801,6.755099
2,MIR6859-1,4.936958,2.753825,3.716039,1.255568,3.519731,0.604129,1.526762,3.800124,8.319123,3.210240,0.713290,4.271262
3,MIR1302-2HG,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,MIR1302-2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## LINCS Level 3

RPKM values converted as described in [Subramanian et al (2017)](https://pmc.ncbi.nlm.nih.gov/articles/PMC5990023/):

_Level 3 L1000 data were used, and the GTEx RNAseq data were quantile normalized, log2 scaled 1+RPKM values_

### Log2 Transform

In [ ]:
# Define timepoint list
list_timepoints = ['6h', '24h']

In [11]:
# Iterate through timepoints
for timepoint in tqdm(list_timepoints, desc = 'Converting RPKM to LINCS Level 3', total = len(list_timepoints)):

    # Load data
    df = pd.read_csv(RPKM / f'{timepoint}_rpkm.csv')
    # Set index
    df.set_index('geneid', inplace = True)
    # Log2 transform
    df = np.log2(df + 1)
    # Reset index
    df.reset_index(inplace = True)
    # Save data
    df.to_csv(LEVEL3 / f'{timepoint}_log2.csv', index = False)

# Show example data
df.head()

Converting RPKM to LINCS Level 3: 100%|██████████| 2/2 [00:00<00:00,  2.17it/s]


,geneid,UI_1,UI_2,UI_3,Halo24h_1,Halo24h_2,Halo24h_3,Nita24h_1,Nita24h_2,Nita24h_3,Paro24h_1,Paro24h_2,Paro24h_3
0,DDX11L1,0.000000,0.000000,0.000000,0.000000,0.148817,0.035437,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,WASH7P,3.054217,2.724736,2.888337,4.400948,4.232020,4.296293,2.954042,2.878589,3.000595,2.905138,3.056722,2.955145
2,MIR6859-1,2.569724,1.908361,2.237576,1.173491,2.176237,0.681790,1.337290,2.263072,3.220194,2.073903,0.776770,2.398148
3,MIR1302-2HG,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,MIR1302-2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


### Quantile Normalisation

Quantile normalisation:
- Ranks log2 transformed values for each condition (column) from highest to lowest
- Calculates the mean value for each rank across all conditions
- Maps that mean value to the gene at that rank in all conditions.

In [17]:
# Iterate through timepoints
for timepoint in tqdm(list_timepoints, desc = 'Applying quantile normalisation', total = len(list_timepoints)):

    # Load data
    df_qn = pd.read_csv(LEVEL3 / f'{timepoint}_log2.csv')
    # Set index
    df_qn.set_index('geneid', inplace = True)
    # Apply quantile normalisation
    df_qn = quantile_normalise(df_qn)
    # Reset index
    df_qn.reset_index(inplace = True)
    # Save data
    df_qn.to_csv(LEVEL3 / f'{timepoint}_quantile.csv', index = False)

# Show example data
df_qn.head()

Applying quantile normalisation: 100%|██████████| 2/2 [00:01<00:00,  1.95it/s]


,geneid,UI_1,UI_2,UI_3,Halo24h_1,Halo24h_2,Halo24h_3,Nita24h_1,Nita24h_2,Nita24h_3,Paro24h_1,Paro24h_2,Paro24h_3
0,DDX11L1,0.000000,0.000000,0.000000,0.000000,0.134000,0.017797,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,WASH7P,3.034148,2.576243,2.808431,4.455850,4.304052,4.370435,2.831894,2.872306,2.978227,2.872306,3.150614,3.037710
2,MIR6859-1,2.527424,1.759102,2.151767,1.291438,2.297112,0.690949,1.234679,2.249735,3.209574,2.033063,0.801580,2.494074
3,MIR1302-2HG,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,MIR1302-2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


### Landmark Genes

Quantile normalised timepoint dataframes are filtered for landmark gene data only.

In [ ]:
# Load landmark data
list_landmark = file_to_list(OUTPUT / 'list_landmark.txt')
# Define timepoints
list_timepoints = ['6h', '24h']

In [ ]:
# Iterate through timepoints
for timepoint in tqdm(list_timepoints, desc = 'Filtering for landmark genes', total = len(list_timepoints)):

    # Load quantile normalised data
    df = pd.read_csv(LEVEL3 / f'{timepoint}_quantile.csv')
    # Filter for landmark genes
    df = df[df['geneid'].isin(list_landmark)]
    # Save data
    df.to_csv(LEVEL3 / f'{timepoint}_level3.csv', index = False)

# Get mapped landmark genes
list_mapped = pd.unique(df['geneid']).tolist()

# Report
num_landmark = len(list_landmark)
num_mapped = len(list_mapped)
percent_mapped = num_mapped / num_landmark * 100
print(f'{percent_mapped:.2f}% of LINCS landmark genes found in RNAseq data ({num_mapped}/{num_landmark})')

# Save data
list_to_file(OUTPUT / 'list_landmark_mapped.txt', list_mapped)
# Show example data
df.head()

96.63% of LINCS landmark genes found in RNAseq data (945/978)


,geneid,UI_1,UI_2,UI_3,Halo24h_1,Halo24h_2,Halo24h_3,Nita24h_1,Nita24h_2,Nita24h_3,Paro24h_1,Paro24h_2,Paro24h_3
173,DFFB,1.287781,1.222952,1.403786,0.193897,0.296793,0.333015,1.235117,1.144845,1.337653,1.316467,1.133495,1.392285
202,ICMT,5.182198,5.108433,5.242634,2.938278,3.188946,3.560451,5.230457,5.107717,5.169421,5.200057,4.990749,5.247967
216,KLHL21,3.901071,4.143036,4.226864,5.496810,5.177492,5.391983,3.678795,3.856762,3.758596,3.679207,3.728988,3.974078
274,CLSTN1,5.235578,5.406302,5.423076,5.110578,5.129633,5.203986,5.671879,5.550649,5.539770,5.492869,5.343123,5.436772
291,DFFA,4.001042,3.813106,3.974078,2.706047,2.965616,2.772726,4.201960,4.074316,4.182751,4.276356,4.510248,4.235669


## LINCS Level 4

LINCS Level 3 data is converted to Level 4 by Z-score normalisation:

$$
MAD_i = median_j (|x_{i,j} - median_i|)
$$

**REST OF DESCRIPTION**

In [ ]:
# Define timepoints
list_timepoints = ['6h', '24h']

In [17]:
# Iterate through timepoints
for timepoint in tqdm(list_timepoints, desc = 'Calculating Level 4 scores', total = len(list_timepoints)):

    # Load data
    df = pd.read_csv(LEVEL3 / f'{timepoint}_level3.csv')
    # Set index
    df.set_index('geneid', inplace = True)

    # Define control columns
    list_controls = [column for column in df.columns if 'UI' in column]
    # Define treatment columns
    list_treatment = [column for column in df.columns if column not in list_controls]

    # Calculate row-wise median for controls
    control_median = df[list_controls].median(axis = 1)
    # Calculate absolute value of deviation from median for each control replicate
    absolute_deviation = df[list_controls].sub(control_median, axis = 0).abs()
    # Calculate median absolute deviation (MAD) per gene
    control_mad = absolute_deviation.median(axis = 1)
    # Calculate global median of control deviation values
    mad_floor = absolute_deviation.median().median()
    # Replace control_mad values lower than mad_floor w/ mad_floor
    effective_mad = control_mad.clip(lower = mad_floor)

    # Calculate Level 4 Z-scores for treatment columns
    for column in list_treatment:
        df[column + '_L4'] = (df[column] - control_median) / (1.4826 * effective_mad)
    
    # Extract data
    df = df[[column for column in df if '_L4' in column]]
    # Reset index
    df.reset_index(inplace = True)
    # Save data
    df.to_csv(LEVEL4 / f'{timepoint}_level4.csv', index = False)

# Show example data
df.head()

Calculating Level 4 scores: 100%|██████████| 2/2 [00:00<00:00, 36.33it/s]


,geneid,Halo24h_1_L4,Halo24h_2_L4,Halo24h_3_L4,Nita24h_1_L4,Nita24h_2_L4,Nita24h_3_L4,Paro24h_1_L4,Paro24h_2_L4,Paro24h_3_L4
0,DFFB,-11.380817,-10.310281,-9.933427,-0.547921,-1.487116,0.518866,0.298444,-1.605195,1.087265
1,ICMT,-25.043042,-22.245488,-18.099352,0.538590,-0.831236,-0.142599,0.199312,-2.136649,0.734015
2,KLHL21,10.892675,8.323393,10.049219,-3.735353,-2.303406,-3.093264,-3.732044,-3.331500,-1.359465
3,CLSTN1,-5.195454,-4.860690,-3.554416,4.665825,2.535991,2.344863,1.520872,-1.109960,0.535328
4,DFFA,-22.277565,-17.717303,-21.106112,4.003581,1.761045,3.666096,5.310623,9.419779,4.595792


## LINCS Level 5

LINCS Level 4 data is converted to Level 5 using MODZ calculation:

**DESCRIPTION**

In [18]:
# Define timepoints
list_timepoints = ['6h', '24h']
# Define treatments
list_treatments = ['Halo', 'Nita', 'Paro']

In [41]:
# Iterate through timepoints
for timepoint in list_timepoints:

    # Load data
    df = pd.read_csv(LEVEL4 / f'{timepoint}_level4.csv')
    # Set index
    df.set_index('geneid', inplace = True)

    # Initialise dataframe
    df5 = pd.DataFrame()

    # Iterate through treatments
    for treatment in list_treatments:

        # Extract data
        df_treatment = df[[column for column in df.columns if treatment in column]]
        
        # Extract values
        treatment_values = df_treatment.values
        # Compute pairwise Spearman correlation between replicates
        corr_matrix, _ = spearmanr(treatment_values, axis = 0)
        # Get number of replicates
        num_replicates = df_treatment.shape[1]
        # Extract values from correlation matrix
        corr_matrix = corr_matrix[:num_replicates, :num_replicates]
        # Apply zero to self correlation
        np.fill_diagonal(corr_matrix, 0)

        # Compute weights as sum of correlations per replicate
        weights = corr_matrix.sum(axis = 0)
        # Normalise weights to sum to 1
        weights = weights / weights.sum()
        # Compute MODZ as weighted average of replicate vectors
        modz = np.dot(treatment_values, weights)
        # Convert to dataframe
        df_results = pd.DataFrame(modz, index = df_treatment.index, columns = [treatment])
        # Reset index
        df_results.reset_index(inplace = True)

        # Check df5 for data
        if len(df5) > 1:
            # Merge data
            df5 = pd.merge(df5, df_results, how = 'left', on = 'geneid')
        else:
            # Concatenate data
            df5 = pd.concat([df5, df_results])

    # Save data
    df5.to_csv(LEVEL5 / f'{timepoint}_level5.csv', index = False)

# Show example data
df5.head()

,geneid,Halo,Nita,Paro
0,DFFB,-10.541844,-0.453518,-0.080225
1,ICMT,-21.797593,-0.271461,-0.413584
2,KLHL21,9.755273,-2.918218,-2.783862
3,CLSTN1,-4.537242,2.871772,0.287400
4,DFFA,-20.367231,3.006912,6.465998
